# 01 · Get the data

Downloads the raw data and saves it in `data/raw/`. Nothing is cleaned here.

| Data | Source | Saved as |
|---|---|---|
| Daily prices (Open, High, Low, Close, Volume) of 6 European defence stocks and the EURO STOXX 50 | Yahoo Finance, via `yfinance` | `data/raw/prices/<name>.csv` |
| Daily VSTOXX (implied volatility of the EURO STOXX 50), control variable | Yahoo Finance, via `yfinance` | `data/raw/vstoxx.csv` |
| Daily Geopolitical Risk index: total, Acts, Threats | Caldara & Iacoviello, matteoiacoviello.com/gpr.htm | `data/raw/gpr_daily.csv` |

**Reproducibility.** Files already in `data/raw/` are *not* downloaded again, so the analysis always uses the same data. Set `REDOWNLOAD = True` only to refresh them on purpose. Prices are downloaded with a fixed end date (`END` in `helpers.py`).

In [1]:
import urllib.request

import pandas as pd
import yfinance as yf

from helpers import TICKERS, START, END, GPR_URL, GPR_XLS, GPR_CSV, VSTOXX_CANDIDATES, VSTOXX_CSV, price_path

REDOWNLOAD = False

## Prices

`auto_adjust=True` returns prices adjusted for splits and dividends, so a stock split does not look like a crash.

In [2]:
for t in TICKERS:
    path = price_path(t)
    if path.exists() and not REDOWNLOAD:
        print(f"{t:10s} already in {path.name}, skipped")
        continue
    p = yf.download(t, start=START, end=END, auto_adjust=True, progress=False)
    if isinstance(p.columns, pd.MultiIndex):        # newer yfinance returns two header levels
        p.columns = p.columns.get_level_values(0)
    p.index.name = "Date"
    p[["Open", "High", "Low", "Close", "Volume"]].to_csv(path)
    print(f"{t:10s} {p.index.min().date()} -> {p.index.max().date()}  {len(p):5d} rows  -> {path.name}")

RHM.DE     already in rheinmetall.csv, skipped
LDO.MI     already in leonardo.csv, skipped
BA.L       already in bae.csv, skipped
HO.PA      already in thales.csv, skipped
SAAB-B.ST  already in saab.csv, skipped
AM.PA      already in dassault.csv, skipped
^STOXX50E  already in stoxx50.csv, skipped


## VSTOXX

Implied volatility of the EURO STOXX 50, used only as a control in a robustness check. Yahoo has listed it under different symbols, so each candidate in `VSTOXX_CANDIDATES` is tried and the longest series is kept. If none works, the robustness check is dropped (see README).

In [3]:
if VSTOXX_CSV.exists() and not REDOWNLOAD:
    print(f"VSTOXX already in {VSTOXX_CSV.name}, skipped")
else:
    best, best_t = None, None
    for t in VSTOXX_CANDIDATES:
        try:
            v = yf.download(t, start=START, end=END, auto_adjust=True, progress=False)
        except Exception as e:
            print(f"{t:10s} failed: {e}")
            continue
        if isinstance(v.columns, pd.MultiIndex):
            v.columns = v.columns.get_level_values(0)
        n = len(v)
        print(f"{t:10s} {n} rows" + (f" ({v.index.min().date()} -> {v.index.max().date()})" if n else ""))
        if n and (best is None or n > len(best)):
            best, best_t = v, t
    if best is None:
        print("No VSTOXX source found: the VSTOXX robustness check will be dropped.")
    else:
        best.index.name = "Date"
        best[["Close"]].to_csv(VSTOXX_CSV)
        print(f"Kept {best_t}: {len(best)} rows -> {VSTOXX_CSV.name}")

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ^V2TX"}}}


$^V2TX: possibly delisted; no timezone found



1 Failed download:


['^V2TX']: possibly delisted; no timezone found


^V2TX      0 rows


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: V2TX.DE"}}}


$V2TX.DE: possibly delisted; no timezone found



1 Failed download:


['V2TX.DE']: possibly delisted; no timezone found


V2TX.DE    0 rows
No VSTOXX source found: the VSTOXX robustness check will be dropped.


## Geopolitical Risk index

The daily file is published as `.xls`. It is converted once to a plain CSV with the three series used in the project:

- `gpr`: total daily index (GPRD)
- `acts`: articles about acts: wars starting, attacks (GPRD_ACT)
- `threats`: articles about threats: tensions, risk of war (GPRD_THREAT)

The index is normalised so that its 1985–2019 average is 100.

In [4]:
if not GPR_XLS.exists() or REDOWNLOAD:
    urllib.request.urlretrieve(GPR_URL, GPR_XLS)
    print("GPR downloaded")

g = pd.read_excel(GPR_XLS)
g["date"] = pd.to_datetime(g["DAY"].astype(str), format="%Y%m%d")
g = g.rename(columns={"GPRD": "gpr", "GPRD_ACT": "acts", "GPRD_THREAT": "threats"})
g = g[["date", "gpr", "acts", "threats"]].dropna()
g.to_csv(GPR_CSV, index=False)
print(f"GPR {g.date.min().date()} -> {g.date.max().date()}  {len(g)} days  -> {GPR_CSV.name}")

GPR 1985-01-01 -> 2026-10-05  15253 days  -> gpr_daily.csv
